In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import (confusion_matrix, classification_report, 
                             roc_curve, auc, precision_recall_fscore_support)

# 1. Ayarlar ve Tohum Sabitleme
seed = 42
torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Eğitim için kullanılan donanım: {device}")

# 2. Veri Hazırlama (Kaggle Yolu)
data_path = "/kaggle/input/sarscov2-ctscan-dataset"

# Pre-trained modeller için standart normalizasyon
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset = datasets.ImageFolder(root=data_path, transform=transform)
class_names = dataset.classes

train_size = int(0.7 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# 3. Model Tanımlamaları

# Model A: Custom CNN
class CustomCNN(nn.Module):
    def __init__(self):
        super(CustomCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Flatten()
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 * 56 * 56, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 2)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

# Model B: ResNet18
def get_resnet():
    model = models.resnet18(weights='DEFAULT')
    model.fc = nn.Linear(model.fc.in_features, 2)
    return model

# Model C: Vision Transformer (ViT)
def get_vit():
    model = models.vit_b_16(weights='DEFAULT')
    model.heads.head = nn.Linear(model.heads.head.in_features, 2)
    return model

# 4. Eğitim ve Metrik Hesaplama Fonksiyonları
def train_model(model, model_name, epochs=50):
    print(f"\n>>> {model_name} Modeli İçin {epochs} Epoch'luk Eğitim Başlıyor...")
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-4)
    
    history = {'train_loss': [], 'val_acc': []}
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            
        model.eval()
        correct = 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                correct += (outputs.argmax(1) == labels).sum().item()
        
        val_acc = correct / len(val_dataset)
        history['train_loss'].append(running_loss / len(train_loader))
        history['val_acc'].append(val_acc)
        
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch [{epoch+1}/{epochs}] - Loss: {history['train_loss'][-1]:.4f} - Val Acc: {val_acc:.4f}")
        
    return model, history

def evaluate_on_test(model, loader):
    model.eval()
    y_true, y_pred, y_probs = [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            probs = torch.softmax(outputs, dim=1)[:, 1]
            preds = outputs.argmax(dim=1)
            y_true.extend(labels.numpy())
            y_pred.extend(preds.cpu().numpy())
            y_probs.extend(probs.cpu().numpy())
    
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary')
    acc = (np.array(y_true) == np.array(y_pred)).mean()
    return y_true, y_pred, y_probs, acc, p, r, f1

# 5. Modelleri Çalıştır ve Verileri Topla
model_configs = {
    "Custom_CNN": CustomCNN(),
    "ResNet18": get_resnet(),
    "ViT_B16": get_vit()
}

performance_summary = []
test_data_results = {}

for name, model_obj in model_configs.items():
    trained_model, history = train_model(model_obj, name, epochs=50)
    y_true, y_pred, y_probs, acc, p, r, f1 = evaluate_on_test(trained_model, test_loader)
    
    test_data_results[name] = {'true': y_true, 'pred': y_pred, 'prob': y_probs, 'history': history}
    performance_summary.append({
        'Model': name, 'Accuracy': acc, 'Precision': p, 'Recall': r, 'F1-Score': f1
    })

# 6. KARŞILAŞTIRMALI GÖRSELLEŞTİRME PANELİ

# A. Başarı Metrikleri Karşılaştırma (SÜTUN ÜSTÜNDE DEĞERLERLE)
df_perf = pd.DataFrame(performance_summary)
df_melted = df_perf.melt(id_vars='Model', var_name='Metrik', value_name='Skor')

plt.figure(figsize=(15, 8))
ax = sns.barplot(data=df_melted, x='Metrik', y='Skor', hue='Model', palette='viridis')

# Sütunların üzerine değerleri yazdıran döngü
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', padding=3, fontsize=10, fontweight='bold')

plt.title("Modeller Arası Performans Karşılaştırması (50 Epoch)", fontsize=14)
plt.ylim(0, 1.15) # Değerlerin sığması için üst sınırı biraz artırdık
plt.legend(loc='lower right', frameon=True, shadow=True)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

# B. Karmaşıklık Matrisleri
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for i, name in enumerate(model_configs.keys()):
    cm = confusion_matrix(test_data_results[name]['true'], test_data_results[name]['pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], 
                xticklabels=class_names, yticklabels=class_names)
    axes[i].set_title(f"{name}\nConfusion Matrix")
plt.tight_layout()
plt.show()

# C. ROC Eğrisi Karşılaştırması
plt.figure(figsize=(10, 8))
for name in model_configs.keys():
    fpr, tpr, _ = roc_curve(test_data_results[name]['true'], test_data_results[name]['prob'])
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc(fpr, tpr):.3f})")

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Modeller Arası ROC Eğrisi ve AUC Skorları')
plt.legend()
plt.grid(True)
plt.show()

# D. Eğitim Kayıpları Karşılaştırması (Loss Curve)
plt.figure(figsize=(12, 6))
for name in model_configs.keys():
    plt.plot(test_data_results[name]['history']['train_loss'], label=f"{name} Train Loss")
plt.title("50 Epoch Boyunca Eğitim Kaybı Değişimi")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

# Final Tablosunu Yazdır
print("\n" + "="*60)
print("50 EPOCH SONUNDA DETAYLI TEST SONUÇLARI")
print("="*60)
print(df_perf.to_string(index=False))